# Multi-Paradigm Bug Fixing

This notebook includes:
- SentencePiece Tokenizer Training
- T5-small Pretraining
- 4 Unique Paradigms for AI Bug Fixing:
    - fine-tuning with pretraining
    - fine-tuning without pretraining
    - RAG
    - zero-shot
- Evaluation and Cross-Analysis of all Paradigms


## 1. Environment Setup
- Install dependencies
- Mount Google Drive if needed
- Set paths
- Import project modules
- Set seed and device


In [2]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/sambennett04/multi-paradigm-bug-fixing.git"
REPO_DIR = Path("/content/multi-paradigm-bug-fixing")

# Clone repo if it doesn't exist, otherwise pull latest
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# Add repo root to Python path (so `from src...` works)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Install dependencies
%pip install -r {REPO_DIR}/requirements.txt

# (Optional) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Debug prints (sanity check)
print("Repo dir:", REPO_DIR)
print("src exists:", (REPO_DIR / "src").exists())
print("data_utils exists:", (REPO_DIR / "src" / "data_utils.py").exists())

Already up to date.


ValueError: Mountpoint must not already contain files

In [ ]:
from pathlib import Path
import sys

# Repo root (always correct after %cd)
PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)

#Add repo to sys path so that imported functions resolve correctly
sys.path.append(str(PROJECT_ROOT))

# Persistent storage (Google Drive)
DRIVE_ROOT = Path("/content/drive/MyDrive/multi-paradigm-bug-fixing")
print("Drive root:", DRIVE_ROOT)

DATA_DIR = DRIVE_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Initialize intermediary and final output folders in mounted google drive
OUTPUTS_DIR = DRIVE_ROOT / "outputs"

TOKENIZER_DIR = OUTPUTS_DIR / "tokenizer"
PRETRAIN_DIR = OUTPUTS_DIR / "pretrain_model"
FINETUNE_A_DIR = OUTPUTS_DIR / "finetune_a"
FINETUNE_B_DIR = OUTPUTS_DIR / "finetune_b"
KNOWLEDGE_BASE_DIR = OUTPUTS_DIR / "knowledge_base"
PREDICTIONS_DIR = OUTPUTS_DIR / "predictions"
RESULTS_DIR = OUTPUTS_DIR / "results"

# Create directories
for path in [
    TOKENIZER_DIR,
    PRETRAIN_DIR,
    FINETUNE_A_DIR,
    FINETUNE_B_DIR,
    KNOWLEDGE_BASE_DIR,
    PREDICTIONS_DIR,
    RESULTS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Outputs will be saved to:", OUTPUTS_DIR)


In [ ]:
import random
import torch

#setting random seed for math.random, torch and torch cuda
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

### Helper Method to Skip Phase if Output Already Populated


In [ ]:
def output_exists(output_file_path):
  if output_file_path.exists():
    print(f"Output file {output_file_path} already exists.")
    return True
  else:
    return False

## 2. Load and Prepare Pretraining Data
- Load CodeSearchNet Java
- Sample ~50K methods
- Extract method bodies
- Preview samples


In [ ]:
import json
from src.data_utils import fetch_pretraining_data_hugging_face, save_pretraining_data, load_pretraining_data

pretraining_save_path = DATA_DIR / "pretraining_methods.txt"

if not output_exists(pretraining_save_path):
  #load only the train split of CodeSearchNet, take random sample of 50,000 methods and extract their bodies
  #Print a summary record of the dataset loading with 3 example methods
  pretraining_methods = fetch_pretraining_data_hugging_face(n_samples=50000, random_seed=42, audit_sample_count=3)

  #cache extracted method bodies in a txt file with one flattened function per line
  #this format is in line with what SentencePieceTrainer expects in the next section
  save_pretraining_data(pretraining_save_path=pretraining_save_path, pretraining_methods= pretraining_methods)

#load flattened pretraining data from save file to guarantee constistency across first and repeat runs
pretraining_methods = load_pretraining_data(pretraining_save_path)

## 3. Train/Load SentencePiece Tokenizer and Load Roberta Tokenizer
- Train SentencePiece tokenizer once
  - MUST include T5-required special tokens
- Save tokenizer
- reload and convert tokenizer to hugging face


In [ ]:
from src.tokenizer_utils import train_tokenizer, load_tokenizer
from transformers import RobertaTokenizer

#train tokenizer and save model/vocab files
corpus_file = DATA_DIR / "pretraining_methods.txt"
model_prefix = TOKENIZER_DIR / "tokenizer"

model_file_exists = output_exists(model_prefix.with_suffix(".model"))
vocab_file_exists = output_exists(model_prefix.with_suffix(".vocab"))

if not model_file_exists and not vocab_file_exists:
  train_tokenizer(corpus_file, model_prefix)

#load trained sentence piece tokenizer as HuggingFace unigram model
sentencepiece_tokenizer = load_tokenizer(model_prefix)

## 4. Build 2 T5-small Model's, One For Each Experiment
- Define T5Config manually
- Initialize model from scratch
- Resize token embeddings
- Verify parameter count


In [ ]:
from src.model_utils import build_t5_model

fresh_model_A = build_t5_model(tokenizer=sentencepiece_tokenizer, embd_hidd_dim = 512, feed_forward_dim = 2048, key_val_proj_dim = 64, num_heads = 8, num_encoder_layers = 6, num_decoder_layers = 6)
fresh_model_B = build_t5_model(tokenizer=sentencepiece_tokenizer, embd_hidd_dim = 512, feed_forward_dim = 2048, key_val_proj_dim = 64, num_heads = 8, num_encoder_layers = 6, num_decoder_layers = 6)

print(f"Fresh Model Shape: {fresh_model_A.shared.weight.shape}")

## 5. Pretraining (For Pipeline A)
- Apply span corruption
- Build pretraining dataset
- Train for 3 epochs
- Save final pretrained model

In [ ]:
from src.pretrain_utils import (
    PretrainConfig,
    build_pretrain_dataset,
    run_pretraining,
    filter_by_length
)
from transformers import get_linear_schedule_with_warmup
from math import ceil

config_exists = output_exists(PRETRAIN_DIR / "config.json")
gen_config_exists = output_exists(PRETRAIN_DIR / "generation_config.json")
model_exists = output_exists(PRETRAIN_DIR / "model.safetensors")

if not (config_exists and gen_config_exists and model_exists):
  pretrain_config = PretrainConfig(
      output_dir=PRETRAIN_DIR,
      min_tokens = 10,
      max_tokens = 512,
      max_samples = 1000, #remove if you would like to use whole pretraining corpus
      corruption_rate = 0.15,
      num_epochs = 3,
      batch_size = 8,
      device = "cuda" if torch.cuda.is_available() else "cpu",
  )

  optimizer = torch.optim.AdamW(
      fresh_model_A.parameters(),
      lr=5e-4,
      weight_decay=0.01,
  )

  effective_pretraining_methods = pretraining_methods
  if pretrain_config.max_samples is not None:
      effective_pretraining_methods = effective_pretraining_methods[:pretrain_config.max_samples]

  #Calculate num training steps so that the scheduler can properly decay learning rate of training
  length_tokenized_filtered_methods = 0
  for method in effective_pretraining_methods:
      token_ids = sentencepiece_tokenizer.encode(method, add_special_tokens=False)
      if filter_by_length(token_ids, pretrain_config=pretrain_config):
          length_tokenized_filtered_methods += 1

  steps_per_epoch = ceil(length_tokenized_filtered_methods / pretrain_config.batch_size)
  num_training_steps = pretrain_config.num_epochs * steps_per_epoch

  scheduler = get_linear_schedule_with_warmup(
      optimizer=optimizer,
      num_warmup_steps=0,
      num_training_steps=num_training_steps,
  )

  fresh_model_A.to(pretrain_config.device)

  run_pretraining(raw_pretraining_methods = pretraining_methods, model = fresh_model_A, optimizer = optimizer, tokenizer = sentencepiece_tokenizer, scheduler = scheduler, pretrain_config = pretrain_config)


## 6. Load Fine-tuning Data
- Load CodeXGLUE bug-fix dataset
- Prepare train / validation / test splits
- Inspect example pair


In [ ]:
from src.data_utils import fetch_finetuning_data_hugging_face, save_finetuning_data, load_finetuning_data

finetuning_train_save_path = DATA_DIR / "finetuning_train_method_pairs.txt"
finetuning_valid_save_path = DATA_DIR / "finetuning_valid_method_pairs.txt"

finetuning_train_data_existis = output_exists(finetuning_train_save_path)
finetuning_valid_data_existis = output_exists(finetuning_valid_save_path)

if not (finetuning_train_data_existis and finetuning_valid_data_existis):

  #fetch the buggy and fixed method pairs from codex glue code refinement train and validation split
  finetuning_train_method_pairs, finetuning_valid_method_pairs = fetch_finetuning_data_hugging_face(audit_sample_count=1)

  #save fixed/buggy pairs as json list of list([[buggy, fixed], [buggy, fixed], ...])
  save_finetuning_data(finetuning_save_path=finetuning_train_save_path, finetuning_method_pairs=finetuning_train_method_pairs)
  save_finetuning_data(finetuning_save_path=finetuning_valid_save_path, finetuning_method_pairs=finetuning_valid_method_pairs)

#load train and validation pairs for fientuning
finetuning_train_method_pairs = load_finetuning_data(finetuning_train_save_path)
finetuning_valid_method_pairs = load_finetuning_data(finetuning_valid_save_path)

## 7i. Finetune Pre-trained t5 Model
- Load final pretrained checkpoint
- build fine-tuning dataset
- Train on bug-fixing task
- Generate and save test predictions


In [ ]:
from math import ceil
from transformers import T5ForConditionalGeneration, get_linear_schedule_with_warmup
from src.finetune_utils import run_finetuning, FinetuneConfig

finetune_config = FinetuneConfig(
    output_dir=FINETUNE_A_DIR,
    max_input_len = 512,
    max_target_len = 512,

    max_train_samples = 1000, #remove or set to none for real experiment
    max_val_samples = 100, #remove or set to none for real experiment
    num_epochs = 1,
    batch_size = 8,
    device = "cuda" if torch.cuda.is_available() else "cpu"
)

# load model pretrained in step 5
pretrained_t5_model = T5ForConditionalGeneration.from_pretrained(str(PRETRAIN_DIR))

optimizer = torch.optim.AdamW(
    pretrained_t5_model.parameters(),
    lr=5e-4,
    weight_decay=0.01,
)

#calculate num_training_steps to be used by the schedulers for both pipelines
#scheduler uses num_training_steps to decide how to change the learning rate over time (linearly decay learning rate toward zero over all training steps)
effective_train_pairs = finetuning_train_method_pairs
if finetune_config.max_train_samples is not None:
    effective_train_pairs = effective_train_pairs[:finetune_config.max_train_samples]

num_train_examples = len(effective_train_pairs)
steps_per_epoch = ceil(num_train_examples / finetune_config.batch_size)
num_training_steps = finetune_config.num_epochs * steps_per_epoch

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

pretrained_t5_model.to(finetune_config.device)

run_finetuning(
    train_pairs=finetuning_train_method_pairs,
    val_pairs=finetuning_valid_method_pairs,
    model=pretrained_t5_model,
    optimizer=optimizer,
    tokenizer=sentencepiece_tokenizer,
    scheduler=scheduler,
    finetune_config=finetune_config,
)

## 7ii. Fine-tune Fresh t5 Model
- Use same tokenizer and config
- Train on bug-fixing task
- Generate and save test predictions


In [ ]:
from math import ceil
from transformers import T5ForConditionalGeneration, get_linear_schedule_with_warmup
from src.finetune_utils import run_finetuning

finetune_config = FinetuneConfig(
    output_dir=FINETUNE_B_DIR,
    max_input_len = 512,
    max_target_len = 512,

    max_train_samples = 1000, #remove or set to none for real experiment
    max_val_samples = 100, #remove or set to none for real experiment
    num_epochs = 1,
    batch_size = 8,
    device = "cuda" if torch.cuda.is_available() else "cpu"
)

optimizer = torch.optim.AdamW(
    fresh_model_B.parameters(),
    lr=5e-4,
    weight_decay=0.01,
)

#calculate num_training_steps to be used by the schedulers for both pipelines
#scheduler uses num_training_steps to decide how to change the learning rate over time (linearly decay learning rate toward zero over all training steps)
effective_train_pairs = finetuning_train_method_pairs
if finetune_config.max_train_samples is not None:
    effective_train_pairs = effective_train_pairs[:finetune_config.max_train_samples]

num_train_examples = len(effective_train_pairs)
steps_per_epoch = ceil(num_train_examples / finetune_config.batch_size)
num_training_steps = finetune_config.num_epochs * steps_per_epoch

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

fresh_model_B.to(finetune_config.device)

run_finetuning(
    train_pairs=finetuning_train_method_pairs,
    val_pairs=finetuning_valid_method_pairs,
    model=fresh_model_B,
    optimizer=optimizer,
    tokenizer=sentencepiece_tokenizer,
    scheduler=scheduler,
    finetune_config=finetune_config,
)


## 8 Initialize QWEN and RAG Knowledge BASE

In [ ]:
from src.llm_utils import (
    LLMConfig,
    build_FAISS_index,
    load_qwen_generator,
    save_knowledge_base,
    train_embedding_model_and_generate_embeddings,
)

#Train Code2Vec-style embedding model on buggy methods and embed the corpus
embedding_model, embeddings, metadata = train_embedding_model_and_generate_embeddings(finetuning_train_method_pairs)

print(f"Retained {len(metadata)} parseable training pairs.")
print(f"Embeddings shape: {embeddings.shape}")

#Build FAISS index
faiss_index = build_FAISS_index(embeddings)
print(f"FAISS index size: {faiss_index.ntotal}")

#Save knowledge base
save_knowledge_base(
    embedding_model=embedding_model,
    index=faiss_index,
    metadata=metadata,
    output_dir=KNOWLEDGE_BASE_DIR,
)
print(f"Knowledge base saved to: {KNOWLEDGE_BASE_DIR}")

#Load generator assets
tokenizer, model = load_qwen_generator()

#Initialize config for later zero-shot / RAG generation
llm_config = LLMConfig(
    embedding_model=embedding_model,
    faiss_index=faiss_index,
    metadata=metadata,
    tokenizer=tokenizer,
    model=model,
    num_samples=1,
    temperature=0.0,
    max_new_tokens=512,
    top_k=3,
)

print("LLMConfig initialized.")


## 10. Evaluation
- Generate on Test Set
- Compute and Compare Select Metrics


### 10i. Generate Results for ___ Test Set using all models and configurations
- load saved t5 models
- fetch codex glue code refinement Test Set
- calculate results for each configuration
- save to dictionary

In [ ]:
from src.data_utils import fetch_test_data_hugging_face
from src.llm_utils import generate_code

pretrained_finetuned_t5 = T5ForConditionalGeneration.from_pretrained(str(FINETUNE_A_DIR))
fresh_finetuned_t5 = T5ForConditionalGeneration.from_pretrained(str(FINETUNE_B_DIR))

pretrained_finetuned_t5.eval()
fresh_finetuned_t5.eval()

test_pairs = fetch_test_data_hugging_face()

predictions = {}

for buggy_code,_ in test_pairs:
  predictions[buggy_code] = {}

  t5_source_text = f"fix bug: {buggy_code}"
  input_ids = tokenizer.encode(t5_source_text, return_tensors="pt")

  #find out where each T5 model lives so I can send its inputs there too
  pretrained_device = next(pretrained_finetuned_t5.parameters()).device
  fresh_device = next(fresh_finetuned_t5.parameters()).device

  with torch.no_grad():
      pretrained_fintuned_output_ids = pretrained_finetuned_t5.generate(input_ids.to(pretrained_device))
      fresh_finetuned_output_ids = fresh_finetuned_t5.generate(input_ids.to(fresh_device))

  #decode raw token outputs so they are human understandable
  predictions[buggy_code]["pretrained_finetuned_t5"] = tokenizer.decode(
      pretrained_fintuned_output_ids[0], skip_special_tokens=True
  )
  predictions[buggy_code]["fresh_finetuned_t5"] = tokenizer.decode(
      fresh_finetuned_output_ids[0], skip_special_tokens=True
  )

  #take only top-1 of top-k generated result as prediction (with current code there is only one generation made)
  predictions[buggy_code]["qwen_zero_shot"] = generate_code(buggy_code, llm_config, mode="zero_shot")[0]
  predictions[buggy_code]["qwen_rag"] = generate_code(buggy_code, llm_config, mode="RAG")[0]



### 10ii. Evaluate on select Metrics and Save Predictions/Results:
- Exact Match
- CodeBLEU
- Create result table
- save predictions and results

In [ ]:
import pandas as pd
from codebleu import calc_codebleu

model_names = [
    "pretrained_finetuned_t5",
    "fresh_finetuned_t5",
    "qwen_zero_shot",
    "qwen_rag",
]

aggregate_metrics = {}
row_records = []

for model_name in model_names:
    references = []
    predictions_for_model = []

    for buggy_code, fixed_code in test_pairs:
        pred = predictions[buggy_code][model_name]
        references.append(fixed_code)
        predictions_for_model.append(pred)

    exact_matches = [
        (pred.strip()) == (ref.strip())
        for pred, ref in zip(predictions_for_model, references)
    ]
    exact_match_score = sum(exact_matches) / len(exact_matches)

    codebleu_result = calc_codebleu(
        references=[references],
        predictions=predictions_for_model,
        lang="java",
    )

    row_records.append({
        "Model": model_name,
        "ExactMatch": round(exact_match_score, 4),
        "CodeBLEU": round(codebleu_result["codebleu"], 4),
    })

results_df = pd.DataFrame(row_records).sort_values(
    by=["ExactMatch", "CodeBLEU"],
    ascending=False,
).reset_index(drop=True)

display(results_df)

predictions_path = PREDICTIONS_DIR / "predictions.json"
results_csv_path = RESULTS_DIR / "evaluation_results.csv"

with open(predictions_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

results_df.to_csv(results_csv_path, index=False)

print(f"Saved predictions to {predictions_path}")
print(f"Saved results table to {results_csv_path}")